# FORELÆSNING 4: OOP-integration workshop

> Integration af OOP, fil-I/O og visualisering til overvågningssystemer for enheder.

### Underviser: Martin Siemienski Andersen mvan@hst.aau.dk

**ST2 - Anvendt Programmering**

In [ ]:
from IPython.display import HTML
import requests

url = "https://raw.githubusercontent.com/AAU-ST2-Programming/all_lectures/refs/heads/main/overview_files/shared_overview_table4.html"
html = requests.get(url, timeout=30).text
HTML(html)


## Workshop-læringsmål

Når du er færdig med øvelserne, bør du kunne:
- Designe et klassehierarki: hvad skal i hver klasse?
- Bruge indkapsling: beskytte data, validere input, eksponere via metoder
- Implementere objektinteraktion: objekter der kalder metoder på andre objekter
- Læse og skrive CSV-filer korrekt
- Parse strukturerede data fra filer tilbage til objekter
- Visualisere flere datasæt side om side
- Bygge og teste en komplet end-to-end pipeline
- Verificere dataintegritet: gem, indlæs, tjek for datatab

# Workshop: Fra enhed til fil
## Objektorienteret dataopsamling i Python

___

# Hvorfor Object-Oriented Programming?

OOP er **ikke et mål i sig selv**.

Det er et værktøj, der hjælper jer med at:

- Strukturere data
- Holde styr på kompleksitet
- Udvide kode uden at ødelægge det eksisterende
- Arbejde systematisk i projekter

I dag:  
> data → struktur → analyse → plot


## OOP i projekter

I projekter vil I ofte:

- Læse data fra filer
- Gemme data over tid
- Analysere resultater
- Vise resultater grafisk

OOP gør det muligt at:
- Holde data samlet
- Genbruge analyse
- Ændre visualisering uden at ændre resten

## Fra at læse data … til at skrive data

At læse data:
- Er passivt
- Data findes allerede

At skrive data:
- Er aktivt
- I bestemmer format og struktur
- Fejl kan koste data

Begge dele er lige vigtige.

## Hvorfor er det vigtigt at kunne skrive til filer?

Når man opsamler data:
- Data findes kun i hukommelsen
- Programmet kan stoppe
- Computeren kan gå ned

Hvis data ikke skrives til disk:
> Data er tabt


___

# Unified Modelling Language, et design værktøj
Vi skal nu snakke lidt omkring `Unified Modelling Language (UML)`

- Det er nyttigt at have et værktøj til at **beskrive både flow og struktur** af et program
- Flowcharts viser **hvordan programmet kører trin for trin**
- UML class diagrams viser **hvilke klasser der findes, og hvordan de samarbejder**
- Sammen giver de et **komplet overblik**, før du skriver kode

> Et godt diagram hjælper dig med at **planlægge programmet**, opdage fejl tidligt og forstå ansvarsområder i OOP.


___

## Hvad er et flowchart?

Et flowchart (flowdiagram) er en grafisk måde at vise **processer og beslutninger** på.  

- Symboler viser handlinger, beslutninger og start/slut
- Pile viser rækkefølge og dataflow
- Bruges til at forstå og planlægge programmer

## Flowchart: typiske symboler

- Oval → start / stop  
- Rektangel → handling / proces  
- Rombe → beslutning (ja/nej)  
- Pile → flow / rækkefølge  

> Flowcharts hjælper med at se **hvordan programmet udfører trin for trin**.

## Flowchart: hvorfor bruge det?

- Visualiserer programlogik uden kode  
- Hjælper med at opdage fejl eller ineffektivitet  
- Bruges som **plan før implementering**  
- Kan bruges både i små og store programmer

## Eksempel: flowchart

```mermaid
flowchart LR
    Start((Start))
    Start --> Read[Read data from USB]
    Read --> Store[Store data in memory]
    Store --> Check{Enough data?}
    Check -- No --> Read
    Check -- Yes --> Save[Save data to file]
    Save --> End((End))
````

> Venstre→højre flow viser **dataflow gennem programmet**.

___

## Hvad er et UML class diagram?

Et UML class diagram viser **struktur og relationer mellem klasser** i et objektorienteret program.

* Hver klasse repræsenteres som en boks
* Boks indeholder:

  * Navn
  * Attributter (data)
  * Metoder (funktioner)
* Pile viser **relationer mellem klasser** (fx “provides data to”)

## UML class diagram: hvorfor bruge det?

* Hjælper med at **forstå programarkitektur**
* Viser ansvar og relationer mellem klasser
* Gør det lettere at designe OOP-programmer
* Bruges som plan før implementering af kode

## UML class diagram eksempel (top-down)

```mermaid
classDiagram
    class USBReader {
        +port
        +baudrate
        +read_line()
        +close()
    }

    class DataHandler {
        -values[]
        +add_value(value)
        +get_all()
    }

    class DataSaver {
        -filename
        +save(values)
    }

    USBReader --> DataHandler 
    DataHandler --> DataSaver 
```

> Top-down layout viser **hierarki og afhængigheder** mellem klasser.
>
> \+ betyder `public`  
> \- betyder `private`  
> \# betyder `protected`  

## Pile og deres mening


![classdiagram_relational_arrows](images/Uml_classes_en.png)

## Relationer mellem klasser

- **Pil med tekst** viser, hvordan klasser interagerer  
- Eksempel: `USBReader --> DataHandler : provides data to`  

- Venstre → højre = data eller kontrolflow  
- Top-down (default classDiagram) = hierarki  

> Relationer viser **hvem der bruger hvem** og **hvem der leverer data til hvem**




## Opsummering: Flow and Class Diagrammer

- Flowchart-symboler → hjælper med at forstå **processer**  
- UML-symboler → hjælper med at forstå **struktur og ansvar**  

Sammen giver de et komplet billede:
- Hvordan programmet **kører trin for trin**  
- Hvilke **klasser arbejder sammen**  

> Begge er værktøjer til planlægning før man skriver kode

___
# Workshop Grundidéen

Vi skal bygge et program, der:

- Modtager data
- Gemmer data midlertidigt
- Skriver data permanent til disk

I stedet for én stor kodeblok,
deler vi opgaven op i **roller**.

## Roller i programmet (overblik)

Vi deler programmet op i roller:

- Reader → får data ind
- DataHandler → holder data i hukommelsen
- DataSaver → skriver data til disk

Hver rolle har ét ansvar.

## Hvad er en “Reader”?

En *Reader* er:

- En klasse der **læser data fra én kilde**
- Den kender **kun** denne kilde
- Den fortolker ikke data

Eksempler:
- USBReader → læser fra serial
- FileReader → læser fra fil
- NetworkReader → læser fra netværk

> En Reader leverer rå data – intet andet.

## Hvad er en “DataHandler”?

En *DataHandler* er:

- En klasse der **ejer data i hukommelsen**
- Den ved, hvordan data gemmes internt
- Den kan tilbyde data til andre dele af programmet

Den:
- Læser ikke fra hardware
- Skriver ikke til disk

> DataHandler er programmets “hukommelse”.

## Hvad er en “DataSaver”?

En *DataSaver* er:

- En klasse der **skriver data til disk**
- Den ved hvordan data skal gemmes
- Den kender ikke datakilden

Eksempler:
- TextFileSaver
- CSVFileSaver

> DataSaver gør data permanente.

## Hvorfor opdele på denne måde?

Fordi det giver:

- Klar struktur
- Færre fejl
- Lettere test
- Lettere udvidelse

Hvis én ting ændrer sig,
skal resten helst ikke gøre det.

## Ingen globale variabler

Globale variabler:

* Er svære at holde styr på
* Gør kode svær at teste
* Skjuler dataflow

I workshoppen:

> Data må kun ligge i objekter

## Programmet som samarbejde

Tænk programmet som et team:

* Reader → henter data
* Handler → holder styr på data
* Saver → gemmer data

`main` er kun:

* Koordinator
* Ikke arbejdshest

___

## Hvad forventes af jer i workshoppen?

I forventes at:

* Følge diagrammerne
* Designe klasser selv
* Implementere trin for trin
* Forklare jeres valg

Det er **helt OK**, hvis ikke alt virker perfekt.

# Øvelser:


<div style="background-color: rgba(255, 253, 231, 0.2); padding: 30px; border-radius: 10px; border-left: 5px solid #fbc02d;">

## Øvelse 1: Sensor Quality Evaluator (1t 45m)

**Scenarie:** Du arbejder for en medicinsk enhedsfabrikant. 
Før en ny sensor kan godkendes til markedet, skal den testes systematisk.

Din opgave: **Byg en `SensorEvaluator` klasse**, der tager målinger fra en sensor og rapporterer på dens kvalitet.

Dit Device skal kunne svare på:
- **Hvor mange målinger var gyldige vs. ugyldige?** (Kvalitetsscore)
- **Hvis der er meget støj, skal sensoren kasseres.** (Accepttest)

Dette bruges til at vurdere: *"Er denne sensor god nok til at sælge til hospitalet?"*


**Note: Du skal ikke lave sensoren selv.**    
* Lave et objekt af `sensor = FakeSensor(...)`. Se i `files/fake_sensor.py`.  
* For at lave en måling: `measurement = sensor.measure()`. Se i `files/fake_sensor.py`.  
* For at teste målingen: `measurement.is_valid()`. Se i `files/fake_sensor.py`.

---

**Vigtigt at forstå:**

Sensorer har en **random failure rate** – nogle gange fejler de, andre gange virker de perfekt. Det er tilfældigt!

**Derfor:**
- **Test hver sensor mange gange** (fx 1000 målinger) for at få et pålideligt gennemsnit
- **Test flere sensorer** af samme type – en god test af én sensor betyder ikke, at alle sensorer er gode
- Som ingeniør skal du teste **mange enheder** for at være sikker på kvaliteten

**Eksempel:** Hvis du kun tester 10 målinger, kan du være uheldig og få 0% fejl (eller 10% fejl) selvom sensoren normalt har 2%. Med 1000 målinger kommer du tættere på den *sande* fejl-rate.

---

**Dine opgaver:**

1. **Design `SensorEvaluator` klasse baseret på OOP principper fra oop_1-3:**
   - **Indkapsling:** Data må ikke tilgås direkte; brug metoder
   - **Klare ansvar:** Hvad ejer klassen? Hvad kan den gøre?
   - **Validering:** Filtrer automatisk ugyldige målinger

   Lav Flow charts of class diagrams for at planlægge din implementering. 
   Kald første iteration "v1", og lav en "v2" efter du har testet den første version.

2. **Implementér:**
   ```python
   import fake_sensor  # Dette er en simuleret sensor, der giver både gyldige og ugyldige målinger
   evaluator = SensorEvaluator(
       sensor=FakeSensor(sensor_id="sensor-v1", sensor_type="PPG"),
       device_id="sensor-v1",
       name="Pulsmåler v1"
   )
   
   evaluator.take_measurements(n=1000)  # Mange målinger for pålidelig statistik!
   report = evaluator.get_quality_report()
   # {
   #   "total_measurements": 1000,
   #   "valid": 980,
   #   "invalid": 20,
   #   "invalid_rate": 0.02,
   #   "quality_score": "PASS" eller "FAIL",
   #   "mean": 75.3,
   #   "min": 42,
   #   "max": 198
   # }
   ```

3. **Godkendelseskriterier:**
   - ✓ **PASS:** Mindre end 5% ugyldige målinger (≤ 0.05)
   - ✗ **FAIL:** Mere end 5% ugyldige målinger (> 0.05)

4. **Test "PPG" sensortype (1000 målinger hver)**
5. **Test 1000 enheder:**
   
   Som ingeniør skal du teste **flere enheder** – ikke bare én!  
   **Spørgsmål:** Får alle 1000 sensorenheder samme fejl-rate?  
   **Spørgsmål:** Hvor mange % af sensorerne består ikke testen? 

---

**OOP Principler du skal bruge:**
- **Encapsulation:** Sensor og målinger ejes af evaluatoren
- **Validering:** Ugyldige data filtreres automatisk
- **Ansvar:** Evaluatoren håndterer rapport, ikke brugeren
- **Genbrugbarhed:** Samme klasse fungerer for alle sensortyper

---

**Refleksionsspørgsmål:**
- Hvorfor er 1000 målinger bedre end 10 målinger?
- Hvis 5 sensorer har mellem 1.8%-2.3% fejl, kan du stole på at de næste 100 sensorer også vil have det?
- Hvad hvis hospitalet kræver <1% fejl? Skal I redesigne sensoren?

</div>

<div style="background-color: rgba(255, 253, 231, 0.2); padding: 30px; border-radius: 10px; border-left: 5px solid #fbc02d;">

## Øvelse 2: Data-system til flere enheder (1t 45m)

**Scenarie:** Hospitalet har flere monitorer (EKG, PPG, temperatur). Byg et system til at styre alle enheder, indsamle data og muliggøre tvær-enheds-analyse.

---

**Vigtigt at forstå:**

Dit system skal håndtere **flere enheder samtidigt** og kunne arbejde med data fra alle sammen. Du laver en **central kontrol** (DataCollector), som:
- Kender alle enhederne
- Indsamler målinger fra hver
- Tillader analyse på tværs af enheder

Dette bruges til: *"Kan vi se mønstre, når vi sammenligner EKG, puls og temperatur samtidigt?"*

**Nøglekoncept:** Objektkomposition – Collectoren *ejer* enhederne, enhederne ejer målingerne.

---

**Dine opgaver:**

1. **Design `DataCollector` klasse baseret på OOP principper:**
   - **Indkapsling:** Enheder tilgås kun gennem metoder
   - **Klare ansvar:** Collectoren styrer alle enheder centralt
   - **Validering:** Målinger gemmes kun hvis de er gyldige

   Lav Flow charts of class diagrams for at planlægge din implementering. 
   Kald første iteration "v1", og lav en "v2" efter du har testet den første version.

2. **Implementér:**
   ```python
   # Opret collector
   collector = DataCollector()
   
   # Opret 3 enheder: EKG, PPG, Temperatur
   ekg = Device(device_id="ekg_001", name="EKG Monitor", min_val=0, max_val=150)
   ppg = Device(device_id="ppg_001", name="Pulsmåler", min_val=40, max_val=200)
   temp = Device(device_id="temp_001", name="Termometer", min_val=35, max_val=41)
   
   # Tilføj til collector
   collector.add_device(ekg)
   collector.add_device(ppg)
   collector.add_device(temp)
   
   # Generér 200 målinger fra hver enhed
   ```

3. **Analysér og visualisér:**
   - **Subplots:** Ét plot per enhed (3 subplots i samme figur)
   - **Statistiske plot:** Histogram, boxplot eller lignende for at sammenligne data
   - **Spørgsmål:** Hvilken enhed har størst variation? Flest afvisninger?
   
4. **Gem alle data:**
   - **CSV-fil:** Alle målinger med kolonner: `device_type, device_id, timestamp, value`
   - **Metadatafil :** Enhedsnavne, intervaller, indsamlingsdato, antal målinger per enhed

5. **Indlæs og verificér:**
   - Læs CSV og metadata tilbage
   - Rekonstruér enheder og målinger
   - Verificér at ingen data er tabt:
     - Sammenlign rækkeantal
     - Kontrolér værdiintervaller
     - Valider timestamps


---

**OOP Principler du skal bruge:**
- **Objektkomposition:** Collector indeholder Enheder; Enheder indeholder Målinger
- **Encapsulation:** Alle enheders data tilgås gennem collector-metoder
- **Ansvar:** Collectoren styrer datapipelinen (indsamling, lagring, rekonstruktion)
- **Genbrugbarhed:** Samme system virker for belibt mange enheder af belibt mange typer

---

**Refleksionsspørgsmål:**
- Hvorfor er det bedre at have én DataCollector i stedet for at styre alle enheder direkte?
- Hvis du skal tilføje en 4. enhed (fx blodtryksmåler), hvad skal ændres i din kode?
- Hvad sker der, hvis CSV'en mangler nogle målinger? Hvordan verificerer du fuldstændighed?
- Kan dit system håndtere enheder med forskellige målefrekvenser (fx EKG hvert sekund, temperatur hver 5. minut)?

</div>

In [ ]:
from IPython.display import HTML
import requests

url = "https://raw.githubusercontent.com/AAU-ST2-Programming/all_lectures/refs/heads/main/overview_files/shared_overview_table.html"
html = requests.get(url, timeout=30).text
HTML(html)